# **Case Study 3: Intensive Care Beds Forecasting. Comparisons amongs LSTM, GRU, BiLSTM, Vanilla Transformers and then we include adjustments by LLMs**

# **All Installations**

In [95]:
%pip install transformers datasets peft accelerate evaluate torch --quiet
%pip install python-dotenv

# **All Imports**

In [96]:
import os
import re
import random
import timeit
import numpy as np
import pandas as pd
from dotenv import load_dotenv
import warnings
from pathlib import Path
import statsmodels.api as sm
from tqdm.notebook import tqdm
from sklearn.metrics import mean_absolute_percentage_error, r2_score, root_mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import plotly.express as px
import matplotlib.pyplot as plt

# Deep Learning Imports
from tensorflow.keras import Input, Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GRU, LSTM, Bidirectional, Conv1D, GlobalAveragePooling1D, Dropout, MultiHeadAttention
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import Adam

# LLMs Imports
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# **Globals**

In [97]:
neural_model = "LSTM"         # LSTM, GRU, BiLSTM

# Dataset
dataset_url = "https://raw.githubusercontent.com/pcm-dpc/COVID-19/master/dati-regioni/dpc-covid19-ita-regioni.csv"
selected_region = "Lazio"

# **All Code, Models and Pictures dowloads**

In [124]:
!cp -rp "./env" ./.env 2>/dev/null
load_dotenv()

True

# **Initializations**

In [99]:
warnings.filterwarnings('ignore')

# **Hyper-parameters**

In [100]:
epochs = 250
batch_size = 256
neural_cells = 600
num_attention_heads = 4

# **Data Download**

In [101]:
df = pd.read_csv(dataset_url)
df.head()

,data,stato,codice_regione,denominazione_regione,lat,long,ricoverati_con_sintomi,terapia_intensiva,totale_ospedalizzati,isolamento_domiciliare,...,note,ingressi_terapia_intensiva,note_test,note_casi,totale_positivi_test_molecolare,totale_positivi_test_antigenico_rapido,tamponi_test_molecolare,tamponi_test_antigenico_rapido,codice_nuts_1,codice_nuts_2
0,2020-02-24T18:00:00,ITA,13,Abruzzo,42.351222,13.398438,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-02-24T18:00:00,ITA,17,Basilicata,40.639471,15.805148,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-02-24T18:00:00,ITA,18,Calabria,38.905976,16.594402,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-02-24T18:00:00,ITA,15,Campania,40.839566,14.250850,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-02-24T18:00:00,ITA,8,Emilia-Romagna,44.494367,11.341721,10,2,12,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [102]:
df_selected = df[df["denominazione_regione"] == selected_region]
df_selected = df_selected.sort_values("data")
y = df_selected["terapia_intensiva"].values.astype(float)

In [103]:
df_selected['data'] = pd.to_datetime(df_selected['data']).dt.floor('D')
df_selected['year_month'] = df_selected['data'].dt.to_period('M')

df_selected = (
    df_selected
    .drop_duplicates(subset=['year_month'], keep='first')
    [['data', 'terapia_intensiva']]
    .rename(columns={'data': 'Date', 'terapia_intensiva': 'y'})
    .reset_index(drop=True)
)

In [104]:
df_selected

,Date,y
0,2020-02-24,1
1,2020-03-01,0
2,2020-04-01,177
3,2020-05-01,105
4,2020-06-01,56
5,2020-07-01,12
6,2020-08-01,9
7,2020-09-01,8
8,2020-10-01,49
9,2020-11-01,185


# **General Functions Definition**

In [105]:
def rmse(y, yhat):
    return root_mean_squared_error(y, yhat)
    #return np.sqrt(np.mean((y - yhat)**2))

def mae(y, yhat):
    return np.mean(np.abs(y - yhat))

def mape(y, yhat):
    # mean_absolute_percentage_error from sklearn returns a fraction, multiply by 100 for percentage
    return mean_absolute_percentage_error(y, yhat)

def r2score(y_true, y_pred):
    return r2_score(y_true, y_pred)

def metrics_compute(metric_name, model_name, model = None, scaler = None, X_dataset = None, dataset_y = None, X_train = None, train_y = None, X_test = None, test_y = None, y_hat = None, y_true = None):

  if metric_name.lower() == "rmse":
    metric = rmse
  elif metric_name.lower() == "mae":
    metric = mae
  elif metric_name.lower() == "mape":
    metric = mape
  else:
    metric = r2_score

  if X_dataset is not None:
    y_hat_data, y_true_data = denormalize_real_and_predicted_output(X_dataset, dataset_y, model, scaler)
    data_metric = metric(y_true_data, y_hat_data)
    print(f"{model_name} Data-set {metric_name}:", round(data_metric, 3))
  else:
    y_hat_data, y_true_data = y_true, y_hat
    data_metric = metric(y_true_data, y_hat_data)
    print(f"{model_name} Data-set {metric_name}:", round(data_metric, 3))

  if X_train is not None:
    y_hat_train, y_true_train = denormalize_real_and_predicted_output(X_train, train_y, model, scaler)
    train_metric = metric(y_true_train, y_hat_train)
    print(f"{model_name} Train-set {metric_name}:", round(train_metric, 3))
  else:
    y_hat_train, y_true_train, train_metric = None, None, None

  if X_test is not None:
    y_hat_test, y_true_test = denormalize_real_and_predicted_output(X_test, test_y, model, scaler)
    test_metric = metric(y_true_test, y_hat_test)
    print(f"{model_name} Test-set {metric_name}:", round(test_metric, 3))
  else:
    y_hat_test, y_true_test, test_metric = None, None, None

  return data_metric, train_metric, test_metric, y_hat_data, y_true_data, y_hat_train, y_true_train, y_hat_test, y_true_test

def adjusted_metrics_compute(metric_name, model_name, y_hat_test, y_true_test):

  if metric_name.lower() == "rmse":
    metric = rmse
  elif metric_name.lower() == "mae":
    metric = mae
  elif metric_name.lower() == "mape":
    metric = mape
  else:
    metric = r2_score

  test_metric = metric(y_true_test, y_hat_test)
  print(f"{model_name} Test-set {metric_name}:", round(test_metric, 3))

  return test_metric

def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):
  '''
  Converts a time series into supervised learning
  '''

  n_vars = 1 if type(data) is list else data.shape[1]
  df = pd.DataFrame(data)
  cols, names = list(), list()
  for i in range(n_in, 0, -1):
    cols.append(df.shift(i))
    names += [('var%d(t-%d)' % (j+1, i)) for j in range(n_vars)]
  for i in range(0, n_out):
    cols.append(df.shift(-i))
    if i == 0:
      names += [('var%d(t)' % (j+1)) for j in range(n_vars)]
    else:
      names += [('var%d(t+%d)' % (j+1, i)) for j in range(n_vars)]
  agg = pd.concat(cols, axis=1)
  agg.columns = names
  if dropnan:
    agg.dropna(inplace=True)
  return agg

def plot_curves(title, neural_model, set_type, denorm_y_pred, denorm_y_test, rmse, mae, mape, r_score):
  pred_df = pd.DataFrame({'Real Y' : denorm_y_test, 'Predicted Y' : denorm_y_pred})

  pred_title = title + "RMSE: "+str(np.round(rmse,3))+' - MAE: '+str(np.round(mae,3))+' - RSCORE: '+str(np.round(r_score,3)) + ' - MAPE: '+str(np.round(mape,3))

  plt.figure()												# generate a new window
  plt.plot(denorm_y_pred, label='Predicted Y', color = 'red')
  plt.plot(denorm_y_test, label='Real Y', color = 'blue')
  plt.legend()
  plt.title(pred_title)
  plt.savefig(f"./{neural_model}_forecasting_model_{set_type}_set_predictions.png", dpi = 600)
  plt.close()

  fig = px.line(pd.DataFrame(pred_df), title=pred_title,
                  labels={
                      "index": "Set Data Point",
                      "value": "Real Y / Predicted Y",
                      "variable": "Real / Predicted"
                  })
  fig.write_html(f"./{neural_model}_forecasting_model_{set_type}_set_predictions.html")
  fig.show()

# **Deep Learning Functions Definition**

In [106]:
def model_definition(X_train, summary = False, neural_cells = 50):
  rnn_input_shape = (X_train.shape[1], X_train.shape[2])
  model = Sequential()
  if neural_model == "GRU":
    model.add(GRU(neural_cells, input_shape = rnn_input_shape))
  elif neural_model == "LSTM":
    model.add(LSTM(neural_cells, input_shape = rnn_input_shape))
  elif neural_model == "BiLSTM":
    model.add(Bidirectional(LSTM(neural_cells), input_shape = rnn_input_shape))
  model.add(Dense(1, activation = "sigmoid"))
  model.compile(loss='mae', optimizer='adam')
  if summary is True:
    model.summary()

  return(model)

def auto_rnn(y_train, neural_model, k, epochs, verbose = 0, neural_cells = 50):
  #  Preprocessing for Deep Learning
  y_train_dl = np.reshape(y_train, (len(y_train), 1))
  y_train_dl.shape

  # Transformation from an unsupervised Time Series to a Supervised Dataset suitable for Deep Learning Supervised Models such as LSTM, GRU, Transformers, etc.
  supervised_full_dataset_df = series_to_supervised(y_train_dl, 1, 1)

  # Normalization
  scaler = MinMaxScaler(feature_range=(0, 1))
  supervised_scaled_full_dataset_df_values = scaler.fit_transform(supervised_full_dataset_df.values)

  # Train-Test Split
  X_dataset = supervised_scaled_full_dataset_df_values[:,:-1]
  y_dataset = supervised_scaled_full_dataset_df_values[:,-1]
  X_train, X_test, y_train, y_test = train_test_split(X_dataset, y_dataset, test_size=0.20, shuffle = False)  # Shuffle = False since we do not have to unstructure temporal dependencies in time series
  X_dataset = X_dataset.reshape((X_dataset.shape[0], 1, X_dataset.shape[1]))
  X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
  X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

  if k == 0:
    model = model_definition(X_train, neural_cells)
  else:
    model = load_model(f"./{neural_model}_{epochs}_epochs_{neural_cells}_neural_cells_last.keras")

  history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_data=(X_test, y_test), verbose=verbose, shuffle=False)

  model.save(f"./{neural_model}_{epochs}_epochs_{neural_cells}_neural_cells_last.keras")

  return model, scaler, X_train, y_train, X_test, y_test, X_dataset, y_dataset

def denormalize_real_and_predicted_output(X_test, y_test, forecasting_model, scaler):
  y_pred = forecasting_model.predict(X_test, verbose = 0)
  X_test_reshaped = X_test.reshape((X_test.shape[0], X_test.shape[2]))
  X_test_reshaped.shape

  denorm_X_test = np.concatenate((X_test_reshaped, y_pred), axis=1)
  denorm_X_test = scaler.inverse_transform(denorm_X_test)
  denorm_y_pred = denorm_X_test[:, denorm_X_test.shape[1]-1]

  y_test = y_test.reshape((len(y_test), 1))

  denorm_X_test = np.concatenate((X_test_reshaped, y_test), axis=1)
  denorm_X_test = scaler.inverse_transform(denorm_X_test)
  denorm_y_test = denorm_X_test[:, denorm_X_test.shape[1]-1]

  return(denorm_y_pred,denorm_y_test)

def denormalize_and_predicted_output(X_test, forecasting_model, scaler):
  X_test = np.reshape(X_test, (len(X_test), 1))
  y_pred = fit.predict(X_test, verbose = 0)
  denorm_y_pred = np.concatenate((X_test, y_pred), axis=1)
  denorm_y_pred = scaler.inverse_transform(denorm_y_pred)

  return denorm_y_pred[:, denorm_y_pred.shape[1]-1], denorm_y_pred[:, denorm_y_pred.shape[1]-1][-1]

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    #Attention
    x = MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=dropout
    )(inputs, inputs)
    x = Dropout(dropout)(x)
    res = x + inputs

    # Feed Forward Part
    x = Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(x)
    x = Dropout(dropout)(x)
    x = Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    return x + res

def build_model(
    input_shape,
    head_size,
    num_heads,
    ff_dim,
    num_transformer_blocks,
    mlp_units,
    dropout=0,
    mlp_dropout=0,
):
    inputs = Input(shape=input_shape)
    x = inputs
    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = GlobalAveragePooling1D(data_format="channels_first")(x)
    for dim in mlp_units:
        x = Dense(dim, activation="relu")(x)
        x = Dropout(mlp_dropout)(x)
    outputs = Dense(1)(x)
    return Model(inputs, outputs)

def model_vt_definition(X_train, summary = False, num_attention_heads = 4):
  input_shape = (X_train.shape[1], X_train.shape[2])

  model_transf = build_model(
            input_shape,
            head_size=128,
            num_heads = num_attention_heads,
            ff_dim=6,
            num_transformer_blocks=6,
            mlp_units=[250],
            mlp_dropout=0.3,
            dropout=0.25,
        )

  model_transf.compile(
            loss="mae",
            optimizer=Adam(learning_rate=1e-4),
        )

  if summary is True:
    model_transf.summary()

  return model_transf

def auto_vt(y_train, k, epochs, verbose = 0, num_attention_heads = 4):
  #  Preprocessing for Deep Learning
  y_train_dl = np.reshape(y_train, (len(y_train), 1))
  y_train_dl.shape

  # Transformation from an unsupervised Time Series to a Supervised Dataset suitable for Deep Learning Supervised Models such as LSTM, GRU, Transformers, etc.
  supervised_full_dataset_df = series_to_supervised(y_train_dl, 1, 1)

  # Normalization
  scaler = MinMaxScaler(feature_range=(0, 1))
  supervised_scaled_full_dataset_df_values = scaler.fit_transform(supervised_full_dataset_df.values)

  # Train-Test Split
  X_dataset = supervised_scaled_full_dataset_df_values[:,:-1]
  y_dataset = supervised_scaled_full_dataset_df_values[:,-1]
  X_train, X_test, y_train, y_test = train_test_split(X_dataset, y_dataset, test_size=0.20, shuffle = False)  # Shuffle = False since we do not have to unstructure temporal dependencies in time series
  X_dataset = X_dataset.reshape((X_dataset.shape[0], 1, X_dataset.shape[1]))
  X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
  X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

  if k == 0:
    model = model_vt_definition(X_train, num_attention_heads)
  else:
    model = load_model(f"./vanilla_transformer_{epochs}_epochs_{num_attention_heads}_num_attention_heads_last.keras")

  history_transf=model.fit(
            X_train,
            y_train,
            validation_data = (X_test, y_test),
            epochs = epochs,
            batch_size = batch_size,
            verbose = verbose,
            shuffle = False,
            )

  model.save(f"./vanilla_transformer_{epochs}_epochs_{num_attention_heads}_num_attention_heads_last.keras")

  return model, scaler, X_train, y_train, X_test, y_test, X_dataset, y_dataset

# **LSTM, GRU, BiLSTM**

In [107]:
start_time = timeit.default_timer()
y_train = y
rnn_model, scaler, X_train, train_y, X_test, test_y, X_dataset, dataset_y = auto_rnn(y_train, neural_model, 0, epochs, verbose = 1, neural_cells = neural_cells)
print("Elapsed LSTM Time: ", timeit.default_timer() - start_time)

Epoch 1/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 145ms/step - loss: 0.3540 - val_loss: 0.4852
Epoch 2/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - loss: 0.3515 - val_loss: 0.4815
Epoch 3/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.3489 - val_loss: 0.4776
Epoch 4/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.3462 - val_loss: 0.4736
Epoch 5/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.3435 - val_loss: 0.4695
Epoch 6/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.3407 - val_loss: 0.4652
Epoch 7/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.3378 - val_loss: 0.4606
Epoch 8/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.3347 - val_loss: 0.4558
Epoch 9/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.3315 - val_loss: 0.4507
Epoch 10/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 0.3281 - val_loss: 0.4454
Epoch 11/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - loss: 0.3246 - val_loss: 0.4398
Epoch 12/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 0.3209 - val_l

## **LSTM, GRU, BiLSTM Performance Evaluation**

In [108]:
for metric in ["RMSE", "MAE", "MAPE", "R2-SCORE"]:
  data_metric_rnn, train_metric_rnn, test_metric_rnn, y_hat_data_rnn, y_true_data_rnn, y_hat_train_rnn, y_true_train_rnn, y_hat_test_rnn, y_true_test_rnn = metrics_compute(metric, neural_model, rnn_model, scaler, X_dataset, dataset_y, X_train, train_y, X_test, test_y)
  if metric == "RMSE":
    data_rmse_rnn = data_metric_rnn
    train_rmse_rnn = train_metric_rnn
    test_rmse_rnn = test_metric_rnn
  elif metric == "MAE":
    data_mae_rnn = data_metric_rnn
    train_mae_rnn = train_metric_rnn
    test_mae_rnn = test_metric_rnn
  elif metric == "MAPE":
    data_mape_rnn = data_metric_rnn
    train_mape_rnn = train_metric_rnn
    test_mape_rnn = test_metric_rnn
  else:
    data_r2_score_rnn = data_metric_rnn
    train_r2_score_rnn = train_metric_rnn
    test_r2_score_rnn = test_metric_rnn

  print("\n")

LSTM Data-set RMSE: 13.8
LSTM Train-set RMSE: 12.64
LSTM Test-set RMSE: 17.698


LSTM Data-set MAE: 12.149
LSTM Train-set MAE: 10.788
LSTM Test-set MAE: 17.596


LSTM Data-set MAPE: 366933950476077.94
LSTM Train-set MAPE: 458667438095096.0
LSTM Test-set MAPE: 6.355


LSTM Data-set R2-SCORE: 0.979
LSTM Train-set R2-SCORE: 0.984
LSTM Test-set R2-SCORE: -38.832




## **Real and Predicted Curves Visualization for RNN**

## **Dataset**

In [109]:
plot_curves("Dataset - ", neural_model, "data", y_hat_data_rnn, y_true_data_rnn, data_rmse_rnn, data_mae_rnn, data_mape_rnn, data_r2_score_rnn)

## **Trainset**

In [110]:
plot_curves("Train-set - ", neural_model, "train", y_hat_train_rnn, y_true_train_rnn, train_rmse_rnn, train_mae_rnn, train_mape_rnn, train_r2_score_rnn)

## **Testset**

In [113]:
  plot_curves("Test-set - ", neural_model, "test", y_hat_test_rnn, y_true_test_rnn, test_rmse_rnn, test_mae_rnn, test_mape_rnn, test_r2_score_rnn)

# **Vanilla Transformer**

In [114]:
start_time = timeit.default_timer()
y_train = y
vt_model, scaler, X_train, train_y, X_test, test_y, X_dataset, dataset_y = auto_vt(y_train, 0, epochs, verbose = 1, num_attention_heads = num_attention_heads)
print("Elapsed LSTM Time: ", timeit.default_timer() - start_time)

Epoch 1/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 40s 3s/step - loss: 0.2401 - val_loss: 0.0063
Epoch 2/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.2225 - val_loss: 0.0089
Epoch 3/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.2116 - val_loss: 0.0199
Epoch 4/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.2016 - val_loss: 0.0324
Epoch 5/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.1972 - val_loss: 0.0432
Epoch 6/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.1929 - val_loss: 0.0507
Epoch 7/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.1877 - val_loss: 0.0553
Epoch 8/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.1849 - val_loss: 0.0582
Epoch 9/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.1830 - val_loss: 0.0600
Epoch 10/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.1817 - val_loss: 0.0612
Epoch 11/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.1799 - val_loss: 0.0621
Epoch 12/250
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.1770 - val_los

## **Vanilla Transformer Predictions Performance Evaluation**

In [115]:
for metric in ["RMSE", "MAE", "MAPE", "R2-SCORE"]:
  data_metric_vt, train_metric_vt, test_metric_vt, y_hat_data_vt, y_true_data_vt, y_hat_train_vt, y_true_train_vt, y_hat_test_vt, y_true_test_vt = metrics_compute(metric, "Vanilla Transformer", vt_model, scaler, X_dataset, dataset_y, X_train, train_y, X_test, test_y)
  if metric == "RMSE":
    data_rmse_vt = data_metric_vt
    train_rmse_vt = train_metric_vt
    test_rmse_vt = test_metric_vt
  elif metric == "MAE":
    data_mae_vt = data_metric_vt
    train_mae_vt = train_metric_vt
    test_mae_vt = test_metric_vt
  elif metric == "MAPE":
    data_mape_vt = data_metric_vt
    train_mape_vt = train_metric_vt
    test_mape_vt = test_metric_vt
  else:
    data_r2_score_vt = data_metric_vt
    train_r2_score_vt = train_metric_vt
    test_r2_score_vt = test_metric_vt

  print("\n")

Vanilla Transformer Data-set RMSE: 51.119
Vanilla Transformer Train-set RMSE: 57.069
Vanilla Transformer Test-set RMSE: 6.223


Vanilla Transformer Data-set MAE: 28.098
Vanilla Transformer Train-set MAE: 33.84
Vanilla Transformer Test-set MAE: 5.131


Vanilla Transformer Data-set MAPE: 236521834556050.9
Vanilla Transformer Train-set MAPE: 295652293195062.94
Vanilla Transformer Test-set MAPE: 2.531


Vanilla Transformer Data-set R2-SCORE: 0.71
Vanilla Transformer Train-set R2-SCORE: 0.672
Vanilla Transformer Test-set R2-SCORE: -3.926




## **Real and Predicted Curves Visualization for Vanilla Transformer**

## **Dataset**

In [116]:
plot_curves("Dataset - ", "vt", "data", y_hat_data_vt, y_true_data_vt, data_rmse_vt, data_mae_vt, data_mape_vt, data_r2_score_vt)

## **Trainset**

In [117]:
plot_curves("Train-set - ", "vt", "train", y_hat_train_vt, y_true_train_vt, train_rmse_vt, train_mae_vt, train_mape_vt, train_r2_score_vt)

## **Testset**

In [120]:
plot_curves("Test-set - ", "vt", "test", y_hat_test_vt, y_true_test_vt, test_rmse_vt, test_mae_vt, test_mape_vt, test_r2_score_vt)

# **Adjustment by Open-Weight LLMs**

In [121]:
# LLMs
llm_model_name = "meta-llama/Llama-3.2-1B-Instruct"             # ok, but not high, it's improving r2_score a little bit performance with penalty 1.3
#llm_model_name = "meta-llama/Llama-3.2-3B-Instruct"             # not ok, it's not generating any output, we need to work on hyperparameters
#llm_model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"        # not ok, less r2_score generated
#llm_model_name = "meta-llama/Meta-Llama-3.3-8B-Instruct"        # not working, maybe token
#llm_model_name = "Qwen/Qwen2.5-3B-Instruct"                      # not ok, a lot less r2_score generated
#llm_model_name = "Qwen/Qwen2.5-7B-Instruct"
#llm_model_name = "Qwen/Qwen2.5-14B-Instruct"
#llm_model_name = "Qwen/Qwen2.5-32B-Instruct"
#llm_model_name = "Qwen/Qwen2.5-72B-Instruct"
#llm_model_name = "mistralai/Mistral-7B-Instruct"

n_trials = 5
differential_corrections = True                   # In this case generate the corrections and not the absolute values of y_hat_test
improvement_delta = 0.07

In [122]:
y_true_train_vt_df = pd.DataFrame(y_true_train_vt)
y_hat_train_vt_df = pd.DataFrame(y_hat_train_vt)
y_hat_test_vt_df = pd.DataFrame(y_hat_test_vt)
y_true_test_vt_df = pd.DataFrame(y_true_test_vt)
y_true_train_vt_df.to_csv("./y_true_train_vt.csv")
y_hat_train_vt_df.to_csv("./y_hat_train_vt.csv")
y_hat_test_vt_df.to_csv("./y_hat_test_vt.csv")
y_true_test_vt_df.to_csv("./y_true_test_vt.csv")

In [125]:
start_time = timeit.default_timer()

for k in range(n_trials):

  train_y = y_true_train_vt.tolist()
  train_pred = y_hat_train_vt.tolist()
  test_pred = y_hat_test_vt.tolist()

  tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
  tokenizer.pad_token = tokenizer.eos_token
  llm_model = AutoModelForCausalLM.from_pretrained(llm_model_name, dtype=torch.float16, device_map="auto")

  # few-shot learning
  examples = []

  for y_true, y_pred in list(zip(train_y, train_pred)):
      examples.append(
          f"Model prediction: {y_pred:.5f}\n"
          f"Correct real value: {y_true:.5f}\n"
      )

  system_instruction = (
      "You are an expert model in time series correction. "
      "You will be shown a model's predictions and the correct real values. "
      "Learn the type of error and how to correct it. "
      "Then you will receive only new predictions and must return "
      "the corrected versions, one per line, NUMBERS ONLY."
  )

  base_prompt = (
      f"{system_instruction}\n\n"
      "CORRECTION EXAMPLES:\n"
      f"{"\n".join(examples)}\n"
  )

  test_values = "\n".join([f"{v:.5f}" for v in test_pred])

  test_prompt = (
      base_prompt +
      "NOW CORRECT THESE NEW PREDICTIONS\n"
      "Model predictions to correct:\n"
      f"{test_values}\n\n"
      "Corrected values:"
  )

  inputs = tokenizer(test_prompt, return_tensors="pt").to(llm_model.device)


  if n_trials == 1:
    with torch.no_grad():
        output = llm_model.generate(
            **inputs,
            max_new_tokens=4096,
            temperature=0.2,
            repetition_penalty=1.3,
            do_sample=False
        )
  else:
    with torch.no_grad():
        output = llm_model.generate(
            **inputs,
            max_new_tokens=4096,
            temperature = random.uniform(0.0, 1.2),
            repetition_penalty = random.uniform(1.0, 1.6),
            do_sample=False
        )

  decoded = tokenizer.decode(output[0], skip_special_tokens=True)

  corrected_part = decoded.split("Corrected values:")[-1].strip()

  numbers = re.findall(r"-?\d+\.\d+|-?\d+", corrected_part)
  numbers = [float(n) for n in numbers]
  numbers = np.array(numbers)
  if len(numbers) < len(test_pred):
    y_hat_test_vt_adjusted = numbers
    y_true_test_vt_for_adj = y_true_test_vt[0:len(numbers)]
  else:
    y_hat_test_vt_adjusted = numbers[0:len(test_pred)]
    y_true_test_vt_for_adj = y_true_test_vt

  if numbers.size > 0:
    print(f"\nTrial {k}: ")
    for metric in ["RMSE", "MAE", "MAPE", "R2-SCORE"]:
      test_metric_vt = adjusted_metrics_compute(metric, "Vanilla Transformer", y_hat_test_vt_adjusted, y_true_test_vt_for_adj)
      if metric == "RMSE":
        test_rmse_vt_adjusted = test_metric_vt
      elif metric == "MAE":
        test_mae_vt_adjusted = test_metric_vt
      elif metric == "MAPE":
        test_mape_vt_adjusted = test_metric_vt
      else:
        test_r2_score_vt_adjusted = test_metric_vt
  else:
    print(f"\nTrial {k}: no values in output")

  if test_r2_score_vt_adjusted > test_r2_score_vt + improvement_delta:
    break

print("Elapsed LLM Time: ", timeit.default_timer() - start_time)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


OutOfMemoryError: CUDA out of memory. Tried to allocate 99.70 GiB. GPU 0 has a total capacity of 14.56 GiB of which 6.69 GiB is free. Including non-PyTorch memory, this process has 7.87 GiB memory in use. Of the allocated memory 7.03 GiB is allocated by PyTorch, and 431.11 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## **Adjusted Test Set Predictions Performance Evaluation**

In [ ]:
plot_curves("Test-set - ", llm_model_name[0:7].replace('/','_'), "test", y_hat_test_vt_adjusted, y_true_test_vt_for_adj, test_rmse_vt_adjusted, test_mae_vt_adjusted, test_mape_vt_adjusted, test_r2_score_vt_adjusted)